In [0]:
# nb_01_ingest_ebird_to_volume.py
#
# Responsibilities:
#   - Call eBird API (checklist index + detail)
#   - Write raw JSONL to a Unity Catalog Volume
#   - No Spark, no Delta — pure Python
#
# Volume path convention:
#   /Volumes/{catalog}/{schema}/{volume}/
#   e.g. /Volumes/birds/ebird_landing/raw/
#
# Unity Catalog Volumes support native Python file I/O via open()
# No mounts, no DBFS, no cloud SDK needed.

import requests
import json
import uuid
import time
import os
from datetime import datetime, date, timedelta

# ── Config ────────────────────────────────────────────────────────────────────
EBIRD_TOKEN     = dbutils.secrets.get(scope="ebird", key="api_key")
REGION          = "US-CA"

# Unity Catalog Volume path — the volume must exist before running
# CREATE VOLUME birds.ebird_landing.raw
VOLUME_ROOT     = "/Volumes/birds/ebird_landing/raw"

PAGE_SIZE       = 200
THROTTLE_S      = 0.5
MAX_RETRIES     = 3
RETRY_BACKOFF   = 2.0

# ── Widgets ───────────────────────────────────────────────────────────────────
def get_widget(name: str, default: str) -> str:
    try:
        val = dbutils.widgets.get(name)
        return val if val.strip() else default
    except Exception:
        return default

yesterday   = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")
RUN_DATE    = get_widget("run_date", yesterday)
RUN_ID      = get_widget("run_id",   str(uuid.uuid4()))
INGEST_TS   = datetime.utcnow().isoformat()
MAX_DETAILS = int(get_widget("max_details", "500"))

print(f"run_date={RUN_DATE} | run_id={RUN_ID}")

# ── HTTP helper ───────────────────────────────────────────────────────────────
def get_with_retry(url: str, params: dict = None) -> requests.Response:
    headers = {"X-eBirdApiToken": EBIRD_TOKEN}
    backoff = RETRY_BACKOFF
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=30)
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 60))
                print(f"Rate limited — waiting {wait}s")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp
        except requests.exceptions.RequestException as e:
            if attempt == MAX_RETRIES:
                raise
            print(f"Attempt {attempt} failed: {e}. Retrying in {backoff}s...")
            time.sleep(backoff)
            backoff *= 2

# ── Volume write helper ───────────────────────────────────────────────────────
def write_jsonl_to_volume(records: list[dict], feed: str) -> str:
    """
    Writes a list of dicts as JSONL to a date-partitioned path inside the volume.
    Returns the full path written.

    Layout:
      /Volumes/birds/ebird_landing/raw/
        checklist_index/
          date=2026-06-06/
            {run_id}.jsonl
        checklist_detail/
          date=2026-06-06/
            {run_id}.jsonl
    """
    year, month, day = RUN_DATE.split("-")
    dir_path  = f"{VOLUME_ROOT}/{feed}/date={RUN_DATE}"
    file_path = f"{dir_path}/{RUN_ID}.jsonl"

    # Volumes support standard Python os + open() — no dbutils.fs needed
    os.makedirs(dir_path, exist_ok=True)

    with open(file_path, "w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

    print(f"Wrote {len(records)} records → {file_path}")
    return file_path

# ── Step 1: Fetch checklist index ─────────────────────────────────────────────
def fetch_checklist_index(region: str, target_date: str) -> list[dict]:
    url      = f"https://api.ebird.org/v2/product/lists/{region}"
    all_rows = []
    page     = 1

    while page <= 50:
        resp      = get_with_retry(url, {"maxResults": PAGE_SIZE, "date": target_date})
        page_data = resp.json()

        if not page_data:
            break

        for row in page_data:
            row["_page_num"] = page
        all_rows.extend(page_data)

        if len(page_data) < PAGE_SIZE:
            break

        page += 1
        time.sleep(0.25)

    print(f"Index: {len(all_rows)} stubs across {page} page(s)")
    return all_rows

index_rows = fetch_checklist_index(REGION, RUN_DATE)

if not index_rows:
    print(f"No checklists for {RUN_DATE} — exiting")
    dbutils.notebook.exit("no_data")

# Wrap each row with ingestion envelope before writing
index_records = [
    {
        "_run_id":       RUN_ID,
        "_ingestion_ts": INGEST_TS,
        "_run_date":     RUN_DATE,
        "_source":       f"ebird/product/lists/{REGION}",
        "_page_num":     row.pop("_page_num"),   # promote out of payload
        "payload":       row,
    }
    for row in index_rows
]

write_jsonl_to_volume(index_records, "checklist_index")

# ── Step 2: Fetch checklist detail ────────────────────────────────────────────
sub_ids = [r["payload"].get("subId") for r in index_records if r["payload"].get("subId")]
print(f"Fetching detail for {len(sub_ids)} checklists...")

# Cap to max_details — remaining sub_ids get picked up by subsequent runs
if len(sub_ids) > MAX_DETAILS:
    print(f"Capping detail fetch: {MAX_DETAILS} of {len(sub_ids)} sub_ids this run")
    sub_ids = sub_ids[:MAX_DETAILS]

detail_records = []

for i, sub_id in enumerate(sub_ids):
    url         = f"https://api.ebird.org/v2/product/checklist/view/{sub_id}"
    http_status = None
    fetch_error = None
    raw_payload = None

    try:
        resp        = get_with_retry(url)
        http_status = resp.status_code
        raw_payload = resp.json()
    except requests.exceptions.HTTPError as e:
        http_status = e.response.status_code if e.response else None
        fetch_error = str(e)
    except Exception as e:
        fetch_error = str(e)

    detail_records.append({
        "_run_id":       RUN_ID,
        "_ingestion_ts": INGEST_TS,
        "_run_date":     RUN_DATE,
        "_source":       f"ebird/product/checklist/view/{sub_id}",
        "_http_status":  http_status,
        "_fetch_error":  fetch_error,
        "payload":       raw_payload,   # None on failure — kept as error record
    })

    time.sleep(THROTTLE_S)

    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(sub_ids)} fetched...")

write_jsonl_to_volume(detail_records, "checklist_detail")

# ── Summary + error gate ──────────────────────────────────────────────────────
ok_count  = sum(1 for r in detail_records if r["_fetch_error"] is None)
err_count = len(detail_records) - ok_count
err_rate  = err_count / len(detail_records) if detail_records else 0

print(f"""
Landing zone drop complete
────────────────────────────────
run_date      : {RUN_DATE}
run_id        : {RUN_ID}
index records : {len(index_records)}
detail ok     : {ok_count}
detail errors : {err_count}
────────────────────────────────
""")

if err_rate > 0.05:
    raise Exception(
        f"Detail fetch error rate {err_rate:.1%} exceeds 5% — "
        f"check landing volume for run_id={RUN_ID}"
    )

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
import csv

# Fetch taxonomy CSV
response = requests.get(
    "https://api.ebird.org/v2/ref/taxonomy/ebird",
    params={"fmt": "csv"},
    headers={"x-ebirdapitoken": dbutils.secrets.get(scope="ebird", key="api_key")}
)
response.raise_for_status()

# Parse CSV lines into rows
lines = response.text.splitlines()
reader = csv.DictReader(lines)
rows = [list(row.values()) for row in reader]
columns = [c.strip().lower().replace(" ", "_").replace("/", "_") for c in reader.fieldnames]

# Create Spark DataFrame directly
df = spark.createDataFrame(rows, schema=columns)

# Overwrite silver table
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("birds.ebird_bronze.ebird_taxonomy")
)

print(f"Written {df.count()} rows to birds.ebird_bronze.ebird_taxonomy")